## 准备数据

In [41]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, optimizers, datasets

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # or any {'0', '1', '2'}

def mnist_dataset():
    (x, y), (x_test, y_test) = datasets.mnist.load_data()
    #normalize
    x = x/255.0
    x_test = x_test/255.0
    
    return (x, y), (x_test, y_test)

In [42]:
print(list(zip([1, 2, 3, 4], ['a', 'b', 'c', 'd'])))

[(1, 'a'), (2, 'b'), (3, 'c'), (4, 'd')]


## 建立模型

In [43]:
class myModel:
    def __init__(self):
        ####################
        '''声明模型对应的参数'''
        ####################
        """
        初始化模型参数
        """
        # 定义模型参数
        self.W1 = tf.Variable(tf.random.normal([784, 256], stddev=0.1), trainable=True)  # 输入层到隐藏层的权重
        self.b1 = tf.Variable(tf.zeros([256]), trainable=True)  # 隐藏层的偏置
        self.W2 = tf.Variable(tf.random.normal([256, 10], stddev=0.1), trainable=True)  # 隐藏层到输出层的权重
        self.b2 = tf.Variable(tf.zeros([10]), trainable=True)  # 输出层的偏置

        

    def __call__(self, x):
        ####################
        '''实现模型函数体，返回未归一化的logits'''
        ####################
        """
        实现模型的前向传播
        - x: 输入数据，形状为 (batch_size, 28, 28)
        - 返回: 未归一化的 logits，形状为 (batch_size, 10)
        """
        # 将输入展平为 (batch_size, 784)
        x = tf.reshape(x, [-1, 784])
        # 第一层全连接层 + ReLU 激活函数
        h1 = tf.nn.relu(tf.matmul(x, self.W1) + self.b1)
        # 第二层全连接层（输出层）
        logits = tf.matmul(h1, self.W2) + self.b2
        return logits
        
model = myModel()

optimizer = optimizers.Adam(learning_rate=0.0001)

## 计算 loss

In [44]:
@tf.function
def compute_loss(logits, labels):
    return tf.reduce_mean(
        tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels))

@tf.function
def compute_accuracy(logits, labels):
    predictions = tf.argmax(logits, axis=1)
    return tf.reduce_mean(tf.cast(tf.equal(predictions, labels), tf.float32))

@tf.function
def train_one_step(model, optimizer, x, y):
    with tf.GradientTape() as tape:
        logits = model(x)
        loss = compute_loss(logits, y)

    # compute gradient
    trainable_vars = [model.W1, model.W2, model.b1, model.b2]
    grads = tape.gradient(loss, trainable_vars)
    for g, v in zip(grads, trainable_vars):
        v.assign_sub(0.01*g)

    accuracy = compute_accuracy(logits, y)

    # loss and accuracy is scalar tensor
    return loss, accuracy

@tf.function
def test(model, x, y):
    logits = model(x)
    loss = compute_loss(logits, y)
    accuracy = compute_accuracy(logits, y)
    return loss, accuracy

## 实际训练

In [46]:
train_data, test_data = mnist_dataset()
for epoch in range(100):
    loss, accuracy = train_one_step(model, optimizer, 
                                    tf.constant(train_data[0], dtype=tf.float32), 
                                    tf.constant(train_data[1], dtype=tf.int64))
    print('epoch', epoch, ': loss', loss.numpy(), '; accuracy', accuracy.numpy())
loss, accuracy = test(model, 
                      tf.constant(test_data[0], dtype=tf.float32), 
                      tf.constant(test_data[1], dtype=tf.int64))

print('test loss', loss.numpy(), '; accuracy', accuracy.numpy())

epoch 0 : loss 1.3349977 ; accuracy 0.6255
epoch 1 : loss 1.3288682 ; accuracy 0.62776667
epoch 2 : loss 1.3228034 ; accuracy 0.62988335
epoch 3 : loss 1.3168024 ; accuracy 0.63196665
epoch 4 : loss 1.310865 ; accuracy 0.634
epoch 5 : loss 1.3049898 ; accuracy 0.63638335
epoch 6 : loss 1.2991757 ; accuracy 0.6386333
epoch 7 : loss 1.2934217 ; accuracy 0.6404167
epoch 8 : loss 1.287727 ; accuracy 0.64213336
epoch 9 : loss 1.2820908 ; accuracy 0.6444167
epoch 10 : loss 1.2765127 ; accuracy 0.6464667
epoch 11 : loss 1.2709919 ; accuracy 0.64828336
epoch 12 : loss 1.2655274 ; accuracy 0.6505167
epoch 13 : loss 1.2601181 ; accuracy 0.65255
epoch 14 : loss 1.2547631 ; accuracy 0.6544667
epoch 15 : loss 1.2494622 ; accuracy 0.6563
epoch 16 : loss 1.2442145 ; accuracy 0.65828335
epoch 17 : loss 1.239019 ; accuracy 0.66036665
epoch 18 : loss 1.2338753 ; accuracy 0.66218334
epoch 19 : loss 1.2287823 ; accuracy 0.66396666
epoch 20 : loss 1.2237399 ; accuracy 0.66581666
epoch 21 : loss 1.2187471 ;